# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What is the most expensive real estate transaction in Pittsburgh in 2025**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-07-06 19:17:20 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-07-06T19:17:20.366807")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'real estate property sales transactions Pittsburgh':

1. **US States Housing Data** (22.8% match)
   Resource: medianhousingvalue
   Resource ID: `644c4e50-f9bb-4fc2-8b11-18702abc7410`
   Dataset ID: `b6326b4d-b8eb-4882-9a4d-e40da17b0f22`
   Format: CSV | Rows: 728 | Cols: 4
   Tags: US housing market, median housing value, real estate trends, state housing data, housing prices, geospatial housing, historical housing data, US states, housing affordability, property values
   This dataset provides information on median housing values across US state
```


In [ ]:
# Step 1: Semantic Search Resources

# Semantic search via Pinecone vector store
# (requires PineconeVectorStore from data_concierge)
from data_concierge.data_layer.connectors.pinecone_store import PineconeVectorStore

store = PineconeVectorStore()
results = store.search_resources('real estate property sales transactions Pittsburgh', n_results=10)
for r in results:
    print(f"{r['dataset_title']} — {r['resource_id']} (score: {r['score']:.2f})")


## Step 2: Search for Datasets

**Search query:** `real estate sales transactions`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 2 datasets matching 'real estate sales transactions'

1. **City-Owned Property (Real Estate Tax Database)**
   ID: `city-owned-property`
   This dataset contains a listing of property owned by the City of Pittsburgh obtained from the City's Real Estate Database. For a more complete listing of City-owned properties obta
   - City-Owned Properties (CSV) [DataStore] ID: `4ff5eb17-e2ad-4818-97c4-8f91fc6b6396`
   - Data Dictionary with Field Defintions (XLSX) [DataStore] ID: `9070809b-54d9-441d-802e-812fcb7c825a`

2. **City Treasury Sales**
   ID: `city-treasury-sales`
   A listing of all the
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'real estate sales transactions', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Search for Datasets

**Search query:** `property sales Allegheny County`

**Result preview:**
```
Found 41 datasets matching 'property sales Allegheny County'

1. **Allegheny County Property Sale Transactions**
   ID: `real-estate-sales`
   This dataset contains data on all Real Property parcels that have sold since 2013 in Allegheny County, PA.  Before doing any market analysis on property sales, check the sales va
   - Property Sales Transactions (CSV) [DataStore] ID: `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
   - Allegheny County Information Portal Real Estate Sales (HTML) ID: `bfdf3a78-7830-425d-afa2-8fbc99c79dcb`
   - Property Dashboard (HTML) ID: `637ab4db-1313-4512-9c2b-6fa659759200`
```


In [ ]:
# Step 3: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'property sales Allegheny County', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 4: SQL Analysis Query

**SQL:**
```sql
SELECT "PARID", "PROPERTYHOUSENUM", "PROPERTYADDRESS", "PROPERTYCITY", "PROPERTYSTATE", "PROPERTYZIP", "MUNIDESC", "SCHOOLDESC", "PRICE", "SALEDATE", "INSTRTYPDESC", "GRANTEE", "GRANTOR", "PROPERTYUNIT", "CLASSDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' AND "PRICE" IS NOT NULL AND "PRICE" != '' ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10
```

**Result preview:**
```
SQL error (HTTP 409): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"query": ["(psycopg2.errors.UndefinedColumn) column \"PROPERTYADDRESS\" does not exist\nLINE 1: ...ELECT * FROM (SELECT \"PARID\", \"PROPERTYHOUSENUM\", \"PROPERTYA...\n                                                             ^\nHINT:  Perhaps you meant to reference the column \"5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1.PROPERTYADDRESSDIR\" or the column \"5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1.PROPERTYADD
```


In [ ]:
# Step 4: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "PARID", "PROPERTYHOUSENUM", "PROPERTYADDRESS", "PROPERTYCITY", "PROPERTYSTATE", "PROPERTYZIP", "MUNIDESC", "SCHOOLDESC", "PRICE", "SALEDATE", "INSTRTYPDESC", "GRANTEE", "GRANTOR", "PROPERTYUNIT", "CLASSDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= \'2025-01-01\' AND "PRICE" IS NOT NULL AND "PRICE" != \'\' ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 5: Load Data from Resource

**Resource ID:** `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
**Limit:** 3

**Result preview:**
```
Resource: 5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1
Total records: 495,780
Loaded: 3
Fields (25): PARID, FULL_ADDRESS, PROPERTYHOUSENUM, PROPERTYFRACTION, PROPERTYADDRESSDIR, PROPERTYADDRESSSTREET, PROPERTYADDRESSSUF, PROPERTYADDRESSUNITDESC, PROPERTYUNITNO, PROPERTYCITY, PROPERTYSTATE, PROPERTYZIP, SCHOOLCODE, SCHOOLDESC, MUNICODE, MUNIDESC, RECORDDATE, SALEDATE, PRICE, DEEDBOOK, DEEDPAGE, SALECODE, SALEDESC, INSTRTYP, INSTRTYPDESC

Sample (3 rows):

           PARID                          FULL_ADDRESS PROPERTYHOUSENUM PROPERTYFRACTION PROPERTYADDRESSDIR PROPERTYADDRESSSTREET PROPERTYADDRESSSUF 
```


In [ ]:
# Step 5: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1', "limit": 3}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 6: SQL Analysis Query

**SQL:**
```sql
SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PROPERTYSTATE", "PROPERTYZIP", "MUNIDESC", "SCHOOLDESC", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' AND "PRICE" IS NOT NULL AND "PRICE" != '' AND CAST("PRICE" AS FLOAT) > 0 ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10
```

**Result preview:**
```
SQL error (HTTP 500): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Internal Server Error", "message": "Internal Server Error"}, "success": false}
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PROPERTYSTATE", "PROPERTYZIP", "MUNIDESC", "SCHOOLDESC", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= \'2025-01-01\' AND "PRICE" IS NOT NULL AND "PRICE" != \'\' AND CAST("PRICE" AS FLOAT) > 0 ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 7: SQL Analysis Query

**SQL:**
```sql
SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10
```

**Result preview:**
```
SQL error (HTTP 403): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Authorization Error", "message": "Access denied: Not authorized to call function CAST"}, "success": false}
```


In [ ]:
# Step 7: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= \'2025-01-01\' ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 8: SQL Analysis Query

**SQL:**
```sql
SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' ORDER BY "PRICE"::float DESC LIMIT 10
```

**Result preview:**
```
SQL: SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' ORDER BY "PRICE"::float DESC LIMIT 10
Rows: 10
Columns: PARID, FULL_ADDRESS, PROPERTYCITY, PRICE, SALEDATE, INSTRTYPDESC, SALEDESC, MUNIDESC

           PARID                           FULL_ADDRESS    PROPERTYCITY PRICE   SALEDATE     INSTRTYPDESC       SALEDESC         MUNIDESC
8000T02782000000  224 VALLEY VIEW DR, OAKDALE, PA 15071         OAKDALE  None 2025-01-21 Mobile Home Sale NOT APPLICABLE  North Fa
```


In [ ]:
# Step 8: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= \'2025-01-01\' ORDER BY "PRICE"::float DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 9: SQL Analysis Query

**SQL:**
```sql
SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' AND "PRICE" IS NOT NULL AND "PRICE" != '' AND "PRICE" != '0' ORDER BY "PRICE"::float DESC LIMIT 10
```

**Result preview:**
```
SQL error (HTTP 500): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Internal Server Error", "message": "Internal Server Error"}, "success": false}
```


In [ ]:
# Step 9: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= \'2025-01-01\' AND "PRICE" IS NOT NULL AND "PRICE" != \'\' AND "PRICE" != \'0\' ORDER BY "PRICE"::float DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 10: SQL Analysis Query

**SQL:**
```sql
SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' AND "PRICE" IS NOT NULL ORDER BY "PRICE"::float DESC LIMIT 10
```

**Result preview:**
```
SQL: SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= '2025-01-01' AND "PRICE" IS NOT NULL ORDER BY "PRICE"::float DESC LIMIT 10
Rows: 10
Columns: PARID, FULL_ADDRESS, PROPERTYCITY, PRICE, SALEDATE, INSTRTYPDESC, SALEDESC, MUNIDESC

           PARID                                FULL_ADDRESS PROPERTYCITY      PRICE   SALEDATE     INSTRTYPDESC             SALEDESC              MUNIDESC
0009P00050000200      1001 LIBERTY AVE, PITTSBURGH, PA 15222   PITTSBURGH 42000000.0 202
```


In [ ]:
# Step 10: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "PARID", "FULL_ADDRESS", "PROPERTYCITY", "PRICE", "SALEDATE", "INSTRTYPDESC", "SALEDESC", "MUNIDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "SALEDATE" >= \'2025-01-01\' AND "PRICE" IS NOT NULL ORDER BY "PRICE"::float DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Excellent! I found the data! Here's the answer:

---

## 🏆 Most Expensive Real Estate Transaction in Pittsburgh Area (2025)

**1001 Liberty Ave, Pittsburgh, PA 15222** — **$42,000,000**

| Detail | Info |
|---|---|
| **Address** | 1001 Liberty Ave, Pittsburgh, PA 15222 |
| **Sale Price** | **$42,000,000** |
| **Sale Date** | August 5, 2025 |
| **Transaction Type** | Special Warranty Deed — Corporation Transfer |
| **Municipality** | 2nd Ward – Pittsburgh |
| **Parcel ID** | 0009P00050000200 |

---

### 🏙️ Top 10 Most Expensive Transactions in Allegheny County (2025)

| # | Address | Price | Date | Type |
|---|---|---|---|---|
| 1 | 1001 Liberty Ave, Pittsburgh | **$42,000,000** | Aug 5, 2025 | Corp Transfer |
| 2 | 5045 Library Rd, Bethel Park | $39,600,000 | May 5, 2025 | Multi-Parcel |
| 3 | 5027–5055 Library Rd, Bethel Park | $39,600,000 | May 5, 2025 | Multi-Parcel |
| 4 | 50 McIntyre Square Dr, Pittsburgh | $33,250,000 | Nov 22, 2025 | Multi-Parcel |
| 5 | 460–466 Nixon Rd, Cheswick | $32,000,000 | Oct 13, 2025 | Corp Transfer |
| 6 | 3011 Smallman St, Pittsburgh | $25,260,400 | Oct 29, 2025 | Corp Transfer |
| 7 | 210–212 Fowler Rd, Warrendale | $24,352,272 | Jan 14, 2025 | Corp Transfer |
| 8–10 | 5305–5335 Baum Blvd, Pittsburgh | $24,120,000 | Dec 22, 2025 | Multi-Parcel |

---

**Key Takeaways:**
- The top transaction at **1001 Liberty Ave** (downtown Pittsburgh, near the convention center/cultural district) was a **corporate transfer** recorded in August 2025 for **$42 million**.
- The **Library Road corridor in Bethel Park** saw a large **$39.6M multi-parcel commercial sale** in May.
- The **Baum Boulevard** properties in Pittsburgh's East End were bundled in a $24.1M multi-parcel deal in December 2025.

📊 *Source: [Allegheny County Property Sale Transactions](https://data.wprdc.org/dataset/real-estate-sales) — Western PA Regional Data Center (WPRDC). Dataset covers all Allegheny County parcel sales since 2013.***

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-07-06

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-07-06 19:17:20
- **Query**: What is the most expensive real estate transaction in Pittsburgh in 2025
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
